<a href="https://colab.research.google.com/github/kunkashyap/Machine-Learning/blob/main/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Regression - Prediction of Continous Values

##Types of Linear Regression
1. Simple Linear Regression
2. Multiple Linear Regression
3. Polynomial Linear Regression

### Simple Linear Regression

In [4]:
!pip install numpy

In [5]:
import numpy as np

In [1]:
!pip install pandas

In [3]:
import pandas as pd
df = pd.read_csv('/content/Housing.csv')
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


missing values

In [6]:
df.isnull().sum()

,0
price,0
area,0
bedrooms,0
bathrooms,0
stories,0
mainroad,0
guestroom,0
basement,0
hotwaterheating,0
airconditioning,0


checking and handling outliers

### Identifying Outliers using the IQR Method

For numerical features, a common method to identify outliers is using the Interquartile Range (IQR). Outliers are typically defined as values that fall below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR`, where Q1 is the first quartile (25th percentile), Q3 is the third quartile (75th percentile), and IQR is `Q3 - Q1`.

In [8]:
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Column '{col}': {len(outliers)} outliers detected.")
    if not outliers.empty:
        display(outliers[[col]].head()) # Display first few outliers if any

Column 'area': 12 outliers detected.


,area
7,16200
10,13200
56,11440
64,11175
66,13200


Column 'bedrooms': 12 outliers detected.


,bedrooms
7,5
28,5
34,5
89,5
112,6


Column 'bathrooms': 1 outliers detected.


,bathrooms
1,4


Column 'stories': 41 outliers detected.


,stories
1,4
6,4
9,4
17,4
26,4


Column 'parking': 12 outliers detected.


,parking
1,3
3,3
47,3
93,3
225,3


### Handling Outliers: Capping

In [9]:
# Columns identified with outliers for capping
cols_to_cap = ['area', 'bedrooms', 'bathrooms']

for col in cols_to_cap:
    print(f"Capping outliers for column: {col}")
    # Calculate 5th and 95th percentiles
    lower_percentile = df[col].quantile(0.05)
    upper_percentile = df[col].quantile(0.95)

    # Apply capping
    df[col] = np.where(df[col] < lower_percentile, lower_percentile, df[col])
    df[col] = np.where(df[col] > upper_percentile, upper_percentile, df[col])

    print(f"  - Lower bound (5th percentile): {lower_percentile:.2f}")
    print(f"  - Upper bound (95th percentile): {upper_percentile:.2f}")
    print(f"  - Values outside this range have been capped.")

print("\nCapping completed for specified columns.")

Capping outliers for column: area
  - Lower bound (5th percentile): 2562.00
  - Upper bound (95th percentile): 9000.00
  - Values outside this range have been capped.
Capping outliers for column: bedrooms
  - Lower bound (5th percentile): 2.00
  - Upper bound (95th percentile): 4.00
  - Values outside this range have been capped.
Capping outliers for column: bathrooms
  - Lower bound (5th percentile): 1.00
  - Upper bound (95th percentile): 2.00
  - Values outside this range have been capped.

Capping completed for specified columns.


### Re-checking Outliers after Capping

In [10]:
# Re-run the outlier detection to see the effect of capping

# Ensure numerical_cols is defined or re-define it if necessary
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('price', errors='ignore')

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Column '{col}': {len(outliers)} outliers detected after capping.")
    if not outliers.empty:
        display(outliers[[col]].head())

Column 'area': 0 outliers detected after capping.
Column 'bedrooms': 0 outliers detected after capping.
Column 'bathrooms': 0 outliers detected after capping.
Column 'stories': 41 outliers detected after capping.


,stories
1,4
6,4
9,4
17,4
26,4


Column 'parking': 12 outliers detected after capping.


,parking
1,3
3,3
47,3
93,3
225,3


### Handling Remaining Outliers: Capping 'stories' and 'parking'

In [11]:
cols_to_cap_additional = ['stories', 'parking']

for col in cols_to_cap_additional:
    print(f"Capping outliers for column: {col}")
    # Calculate 5th and 95th percentiles
    lower_percentile = df[col].quantile(0.05)
    upper_percentile = df[col].quantile(0.95)

    # Apply capping
    df[col] = np.where(df[col] < lower_percentile, lower_percentile, df[col])
    df[col] = np.where(df[col] > upper_percentile, upper_percentile, df[col])

    print(f"  - Lower bound (5th percentile): {lower_percentile:.2f}")
    print(f"  - Upper bound (95th percentile): {upper_percentile:.2f}")
    print(f"  - Values outside this range have been capped.")

print("\nCapping completed for 'stories' and 'parking'.")

Capping outliers for column: stories
  - Lower bound (5th percentile): 1.00
  - Upper bound (95th percentile): 4.00
  - Values outside this range have been capped.
Capping outliers for column: parking
  - Lower bound (5th percentile): 0.00
  - Upper bound (95th percentile): 2.00
  - Values outside this range have been capped.

Capping completed for 'stories' and 'parking'.


### Final Re-check of Outliers After All Capping

In [12]:
# Final re-run of the outlier detection for all numerical columns

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('price', errors='ignore')

print("Checking for outliers in all numerical columns after all capping operations:")
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Column '{col}': {len(outliers)} outliers detected.")
    if not outliers.empty:
        display(outliers[[col]].head())

Checking for outliers in all numerical columns after all capping operations:
Column 'area': 0 outliers detected.
Column 'bedrooms': 0 outliers detected.
Column 'bathrooms': 0 outliers detected.
Column 'stories': 41 outliers detected.


,stories
1,4.0
6,4.0
9,4.0
17,4.0
26,4.0


Column 'parking': 0 outliers detected.


encoding

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['mainroad'] = le.fit_transform(df['mainroad'])
df['guestroom'] = le.fit_transform(df['guestroom'])
df['basement'] = le.fit_transform(df['basement'])
df['hotwaterheating'] = le.fit_transform(df['hotwaterheating'])
df['airconditioning'] = le.fit_transform(df['airconditioning'])
df['prefarea'] = le.fit_transform(df['prefarea'])
df['furnishingstatus'] = le.fit_transform(df['furnishingstatus'])
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420.0,4.0,2.0,3.0,1,0,0,0,1,2.0,1,0
1,12250000,8960.0,4.0,2.0,4.0,1,0,0,0,1,2.0,0,0
2,12250000,9000.0,3.0,2.0,2.0,1,0,1,0,0,2.0,1,1
3,12215000,7500.0,4.0,2.0,2.0,1,0,1,0,1,2.0,1,0
4,11410000,7420.0,4.0,1.0,2.0,1,1,1,0,1,2.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000.0,2.0,1.0,1.0,1,0,1,0,0,2.0,0,2
541,1767150,2562.0,3.0,1.0,1.0,0,0,0,0,0,0.0,0,1
542,1750000,3620.0,2.0,1.0,1.0,1,0,0,0,0,0.0,0,2
543,1750000,2910.0,3.0,1.0,1.0,0,0,0,0,0,0.0,0,0


In [17]:
y =df['price']
y

,price
0,13300000
1,12250000
2,12250000
3,12215000
4,11410000
...,...
540,1820000
541,1767150
542,1750000
543,1750000


In [20]:
x = df.drop(columns=['price'],axis=1) #coloumn
x

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,7420.0,4.0,2.0,3.0,1,0,0,0,1,2.0,1,0
1,8960.0,4.0,2.0,4.0,1,0,0,0,1,2.0,0,0
2,9000.0,3.0,2.0,2.0,1,0,1,0,0,2.0,1,1
3,7500.0,4.0,2.0,2.0,1,0,1,0,1,2.0,1,0
4,7420.0,4.0,1.0,2.0,1,1,1,0,1,2.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
540,3000.0,2.0,1.0,1.0,1,0,1,0,0,2.0,0,2
541,2562.0,3.0,1.0,1.0,0,0,0,0,0,0.0,0,1
542,3620.0,2.0,1.0,1.0,1,0,0,0,0,0.0,0,2
543,2910.0,3.0,1.0,1.0,0,0,0,0,0,0.0,0,0


In [23]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.2, random_state=42)


In [ ]:
# 20 percent of data will be used for testing purpose and 80 percent will be used in training purpose

In [27]:
from sklearn.linear_model import LinearRegression
le=LinearRegression()
model = le.fit(x_train,y_train) #used for training model
model

LinearRegression()

In [32]:
y_predict = model.predict(x_test)
y_test


,price
316,4060000
77,6650000
360,3710000
90,6440000
493,2800000
...,...
138,5740000
62,7070000
135,5775000
128,5873000


In [33]:
y_predict

array([6061603.20052193, 6635067.60229273, 3195795.76780537,
       4147379.21329699, 3608076.13706472, 3838348.82935216,
       6381946.71881214, 6990563.64670861, 3293051.93094755,
       2858076.63527891, 8261321.59674533, 2770136.48483508,
       2630908.64619437, 3708176.1443445 , 3929161.07468871,
       4590036.55828012, 2975709.54411727, 4566352.86570242,
       5055473.56385597, 3909092.82908628, 4415014.34547343,
       5904208.09352344, 2666417.26929168, 4946069.32390045,
       6440313.15498626, 8267973.22659084, 3861598.37215777,
       6200193.625203  , 7660087.03333976, 3522192.61501881,
       5794182.62462173, 3288138.95235876, 7066063.07133269,
       4505017.48896236, 3502620.18469806, 5669011.13044493,
       6071320.60749851, 3926074.15518741, 2597877.67059101,
       4778372.34438588, 5568239.14081101, 2889606.60066934,
       6605953.43652616, 4455346.21686659, 4182038.13314786,
       4152391.74186739, 6371975.41540546, 4282702.63694947,
       4268060.31352501,